In [0]:
# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ============================================================
# 1. CONFIGURATION
# ============================================================

CATALOG = "aml_engine"
SCHEMA = "aml_poc"

SILVER_TX_TABLE = f"{CATALOG}.{SCHEMA}.silver_transactions"

RULE_RESULTS_TABLE = f"{CATALOG}.{SCHEMA}.rule_results"
RULE_SCORE_TABLE = f"{CATALOG}.{SCHEMA}.rule_transaction_scores"

S3_BASE_PATH = "s3://zubair-s3-demo/raw_dataset/aml"
S3_DELTA_PATH = f"{S3_BASE_PATH}/delta_tables"

RULE_RESULTS_PATH = (
    f"{S3_DELTA_PATH}/rule_results"
)

RULE_SCORE_PATH = (
    f"{S3_DELTA_PATH}/rule_transaction_scores"
)

RULE_CHECKPOINT_PATH = (
    f"{S3_BASE_PATH}/checkpoints/rule_engine"
)


# ============================================================
# 2. RULE PARAMETERS
# ============================================================

HIGH_VALUE_THRESHOLD = 1000.0

VELOCITY_TIME_WINDOW = 2
VELOCITY_THRESHOLD = 5

FAN_IN_THRESHOLD = 5
FAN_OUT_THRESHOLD = 5

RULE_VERSION = "v1.0"


print("Incremental Rule Engine configuration loaded.")
print(f"Silver source: {SILVER_TX_TABLE}")


# COMMAND ----------

# ============================================================
# 3. READ SILVER INCREMENTALLY
# ============================================================
#
# Silver is already incremental.
#
# This stream receives only records newly appended to Silver.
# ============================================================

new_transactions_stream = (
    spark.readStream
    .format("delta")
    .table(SILVER_TX_TABLE)
)


# COMMAND ----------

# ============================================================
# 4. RULE ENGINE FUNCTION
# ============================================================

def process_rule_batch(new_transactions_df, batch_id):

    # --------------------------------------------------------
    # Do not use cache(), persist(), or unpersist().
    # Serverless compute does not support persistence.
    # --------------------------------------------------------

    if new_transactions_df.isEmpty():
        print(f"Batch {batch_id}: no new transactions.")
        return

    print(
        f"Processing Rule Engine batch: {batch_id}"
    )


    # ========================================================
    # 5. RULE R001 - HIGH VALUE TRANSACTION
    # ========================================================

    rule_high_value = (
        new_transactions_df

        .select(
            "tx_id",
            "sender_account_id",
            "receiver_account_id",
            "tx_amount",
            "event_time"
        )

        .withColumn(
            "rule_id",
            F.lit("R001")
        )

        .withColumn(
            "rule_name",
            F.lit("HIGH_VALUE_TRANSACTION")
        )

        .withColumn(
            "rule_category",
            F.lit("AMOUNT")
        )

        .withColumn(
            "rule_triggered",
            F.col("tx_amount") > HIGH_VALUE_THRESHOLD
        )

        .withColumn(
            "rule_score",
            F.when(
                F.col("rule_triggered"),
                F.lit(30)
            ).otherwise(
                F.lit(0)
            )
        )

        .withColumn(
            "rule_evidence",
            F.when(
                F.col("rule_triggered"),
                F.concat(
                    F.lit("Transaction amount "),
                    F.col("tx_amount").cast("string"),
                    F.lit(" exceeds threshold "),
                    F.lit(str(HIGH_VALUE_THRESHOLD))
                )
            ).otherwise(
                F.lit(None).cast("string")
            )
        )
    )


    # ========================================================
    # 6. CREATE RELEVANT TIME WINDOWS
    # ========================================================

    relevant_event_times = (
        new_transactions_df

        .select(
            "event_time"
        )

        .filter(
            F.col("event_time").isNotNull()
        )

        .distinct()

        .withColumn(
            "min_event_time",
            F.col("event_time") -
            F.lit(VELOCITY_TIME_WINDOW)
        )

        .withColumn(
            "max_event_time",
            F.col("event_time")
        )

        .withColumnRenamed(
            "event_time",
            "reference_event_time"
        )
    )


    # ========================================================
    # 7. READ HISTORICAL CONTEXT
    # ========================================================
    #
    # Historical transactions are needed because a new
    # transaction must be evaluated against previous activity.
    #
    # Aliases prevent ambiguous event_time columns.
    # ========================================================

    historical_context = (
        spark.table(SILVER_TX_TABLE).alias("s")

        .join(
            relevant_event_times.alias("r"),

            (
                (F.col("s.event_time") >=
                 F.col("r.min_event_time"))
                &
                (F.col("s.event_time") <=
                 F.col("r.max_event_time"))
            ),

            "inner"
        )

        .select(
            F.col("s.tx_id").alias("tx_id"),
            F.col("s.sender_account_id").alias(
                "sender_account_id"
            ),
            F.col("s.receiver_account_id").alias(
                "receiver_account_id"
            ),
            F.col("s.tx_amount").alias("tx_amount"),
            F.col("s.event_time").alias("event_time")
        )

        .dropDuplicates(
            ["tx_id"]
        )
    )


    # ========================================================
    # 8. CURRENT + HISTORICAL CONTEXT
    # ========================================================

    context_transactions = (
        historical_context

        .unionByName(
            new_transactions_df.select(
                "tx_id",
                "sender_account_id",
                "receiver_account_id",
                "tx_amount",
                "event_time"
            ),
            allowMissingColumns=True
        )

        .dropDuplicates(
            ["tx_id"]
        )
    )


    # ========================================================
    # 9. RULE R002 - TRANSACTION VELOCITY
    # ========================================================

    velocity_window = (
        Window

        .partitionBy(
            "sender_account_id"
        )

        .orderBy(
            "event_time"
        )

        .rangeBetween(
            -VELOCITY_TIME_WINDOW,
            0
        )
    )


    velocity_df = (
        context_transactions

        .withColumn(
            "velocity_count",
            F.count("tx_id").over(
                velocity_window
            )
        )
    )


    rule_velocity = (
        velocity_df

        .join(
            new_transactions_df.select(
                "tx_id"
            ),
            on="tx_id",
            how="inner"
        )

        .select(
            "tx_id",
            "sender_account_id",
            "receiver_account_id",
            "tx_amount",
            "event_time",
            "velocity_count"
        )

        .withColumn(
            "rule_id",
            F.lit("R002")
        )

        .withColumn(
            "rule_name",
            F.lit("TRANSACTION_VELOCITY")
        )

        .withColumn(
            "rule_category",
            F.lit("VELOCITY")
        )

        .withColumn(
            "rule_triggered",
            F.col("velocity_count") >= VELOCITY_THRESHOLD
        )

        .withColumn(
            "rule_score",
            F.when(
                F.col("rule_triggered"),
                F.lit(20)
            ).otherwise(
                F.lit(0)
            )
        )

        .withColumn(
            "rule_evidence",
            F.when(
                F.col("rule_triggered"),
                F.concat(
                    F.lit("Sender executed "),
                    F.col("velocity_count").cast("string"),
                    F.lit(
                        " transactions within the configured time window"
                    )
                )
            ).otherwise(
                F.lit(None).cast("string")
            )
        )
    )


    # ========================================================
    # 10. RULE R003 - FAN IN
    # ========================================================

    fan_in_df = (
        context_transactions

        .groupBy(
            "receiver_account_id",
            "event_time"
        )

        .agg(
            F.countDistinct(
                "sender_account_id"
            ).alias("unique_senders"),

            F.sum(
                "tx_amount"
            ).alias("total_inflow")
        )
    )


    rule_fan_in = (
        new_transactions_df

        .join(
            fan_in_df,
            on=[
                "receiver_account_id",
                "event_time"
            ],
            how="left"
        )

        .select(
            "tx_id",
            "sender_account_id",
            "receiver_account_id",
            "tx_amount",
            "event_time",
            "unique_senders",
            "total_inflow"
        )

        .withColumn(
            "unique_senders",
            F.coalesce(
                F.col("unique_senders"),
                F.lit(0)
            )
        )

        .withColumn(
            "rule_id",
            F.lit("R003")
        )

        .withColumn(
            "rule_name",
            F.lit("FAN_IN")
        )

        .withColumn(
            "rule_category",
            F.lit("NETWORK")
        )

        .withColumn(
            "rule_triggered",
            F.col("unique_senders") >= FAN_IN_THRESHOLD
        )

        .withColumn(
            "rule_score",
            F.when(
                F.col("rule_triggered"),
                F.lit(25)
            ).otherwise(
                F.lit(0)
            )
        )

        .withColumn(
            "rule_evidence",
            F.when(
                F.col("rule_triggered"),
                F.concat(
                    F.lit("Receiver received funds from "),
                    F.col("unique_senders").cast("string"),
                    F.lit(
                        " distinct senders at the same event time"
                    )
                )
            ).otherwise(
                F.lit(None).cast("string")
            )
        )
    )


    # ========================================================
    # 11. RULE R004 - FAN OUT
    # ========================================================

    fan_out_df = (
        context_transactions

        .groupBy(
            "sender_account_id",
            "event_time"
        )

        .agg(
            F.countDistinct(
                "receiver_account_id"
            ).alias("unique_receivers"),

            F.sum(
                "tx_amount"
            ).alias("total_outflow")
        )
    )


    rule_fan_out = (
        new_transactions_df

        .join(
            fan_out_df,
            on=[
                "sender_account_id",
                "event_time"
            ],
            how="left"
        )

        .select(
            "tx_id",
            "sender_account_id",
            "receiver_account_id",
            "tx_amount",
            "event_time",
            "unique_receivers",
            "total_outflow"
        )

        .withColumn(
            "unique_receivers",
            F.coalesce(
                F.col("unique_receivers"),
                F.lit(0)
            )
        )

        .withColumn(
            "rule_id",
            F.lit("R004")
        )

        .withColumn(
            "rule_name",
            F.lit("FAN_OUT")
        )

        .withColumn(
            "rule_category",
            F.lit("NETWORK")
        )

        .withColumn(
            "rule_triggered",
            F.col("unique_receivers") >= FAN_OUT_THRESHOLD
        )

        .withColumn(
            "rule_score",
            F.when(
                F.col("rule_triggered"),
                F.lit(25)
            ).otherwise(
                F.lit(0)
            )
        )

        .withColumn(
            "rule_evidence",
            F.when(
                F.col("rule_triggered"),
                F.concat(
                    F.lit("Sender transferred funds to "),
                    F.col("unique_receivers").cast("string"),
                    F.lit(
                        " distinct receivers at the same event time"
                    )
                )
            ).otherwise(
                F.lit(None).cast("string")
            )
        )
    )


    # ========================================================
    # 12. COMBINE ALL RULES
    # ========================================================

    rule_results_df = (
        rule_high_value

        .unionByName(
            rule_velocity,
            allowMissingColumns=True
        )

        .unionByName(
            rule_fan_in,
            allowMissingColumns=True
        )

        .unionByName(
            rule_fan_out,
            allowMissingColumns=True
        )
    )


    # ========================================================
    # 13. ADD COMMON METADATA
    # ========================================================

    rule_results_df = (
        rule_results_df

        .withColumn(
            "rule_execution_timestamp",
            F.current_timestamp()
        )

        .withColumn(
            "rule_version",
            F.lit(RULE_VERSION)
        )

        .withColumn(
            "batch_id",
            F.lit(int(batch_id))
        )
    )


    # ========================================================
    # 14. KEEP ONLY TRIGGERED RULES
    # ========================================================

    triggered_rule_results_df = (
        rule_results_df

        .filter(
            F.col("rule_triggered") == True
        )
    )


    # ========================================================
    # 15. SAVE RULE RESULTS
    # ========================================================
    #
    # append = new rule events only
    #
    # txnAppId + txnVersion provide Delta transaction identity
    # for this batch.
    # ========================================================

    (
        triggered_rule_results_df

        .write

        .format("delta")

        .mode("append")

        .option(
            "path",
            RULE_RESULTS_PATH
        )

        .option(
            "txnAppId",
            "aml_rule_engine"
        )

        .option(
            "txnVersion",
            int(batch_id)
        )

        .saveAsTable(
            RULE_RESULTS_TABLE
        )
    )


    # ========================================================
    # 16. TRANSACTION-LEVEL RULE SCORE
    # ========================================================

    transaction_rule_scores = (
        triggered_rule_results_df

        .groupBy(
            "tx_id"
        )

        .agg(
            F.sum(
                "rule_score"
            ).alias("rule_score"),

            F.collect_set(
                "rule_id"
            ).alias("triggered_rule_ids"),

            F.collect_set(
                "rule_name"
            ).alias("triggered_rules")
        )

        .withColumn(
            "rule_execution_timestamp",
            F.current_timestamp()
        )

        .withColumn(
            "rule_version",
            F.lit(RULE_VERSION)
        )

        .withColumn(
            "batch_id",
            F.lit(int(batch_id))
        )
    )


    # ========================================================
    # 17. SAVE TRANSACTION RULE SCORES
    # ========================================================

    (
        transaction_rule_scores

        .write

        .format("delta")

        .mode("append")

        .option(
            "path",
            RULE_SCORE_PATH
        )

        .option(
            "txnAppId",
            "aml_rule_score"
        )

        .option(
            "txnVersion",
            int(batch_id)
        )

        .saveAsTable(
            RULE_SCORE_TABLE
        )
    )


    print(
        f"Rule Engine batch {batch_id} completed successfully."
    )


# COMMAND ----------

# ============================================================
# 18. START RULE ENGINE
# ============================================================

query = (
    new_transactions_stream

    .writeStream

    .foreachBatch(
        process_rule_batch
    )

    .option(
        "checkpointLocation",
        RULE_CHECKPOINT_PATH
    )

    .trigger(
        availableNow=True
    )

    .start()
)


# COMMAND ----------

# ============================================================
# 19. WAIT FOR COMPLETION
# ============================================================

query.awaitTermination()

print("==============================================")
print("INCREMENTAL RULE ENGINE COMPLETED")
print("==============================================")
print(f"Rule Results Table : {RULE_RESULTS_TABLE}")
print(f"Rule Score Table   : {RULE_SCORE_TABLE}")
print(f"Checkpoint         : {RULE_CHECKPOINT_PATH}")